In [ ]:
from pathlib import Path
from tqdm import tqdm
from asapdiscovery.docking.openeye import POSITDockingResults
from asapdiscovery.data.schema.ligand import Ligand
from asapdiscovery.data.readers.molfile import MolFileFactory
from asapdiscovery.data.backend.openeye import oechem
import argparse
import pandas as pd
from asapdiscovery.docking.docking_data_validation import DockingResultCols

In [ ]:
def calculate_ligand_rmsd(ref: Ligand, fit: Ligand, append_rsmd=True) -> Ligand:
    from asapdiscovery.data.backend.openeye import oechem

    fitmol = fit.to_oemol()
    refmol = ref.to_oemol()
    nConfs = fit.num_poses
    vecRmsd = oechem.OEDoubleArray(nConfs)
    success = oechem.OERMSD(refmol, fitmol, vecRmsd)
    if not success:
        print("RMSD calculation failed")

    if append_rsmd:
        fit.set_SD_data({"RMSD": list(vecRmsd)})
    return fit

In [ ]:
def calculate_ligand_rmsd_oemol(ref: oechem.OEMol, fit: oechem.OEMol) -> float:
    return oechem.OERMSD(ref, fit)

In [ ]:
def get_filtered_poses(posed_ligands: list[Ligand], cutoff):
    """
    Filter out poses with RMSD above cutoff.
    Heavily based on code by Benjamin Kaminow.
    """

    # sort by pose id
    posed_ligands.sort(key=lambda x: x.tags["Pose_ID"])
    all_oemols = [posed_ligand.to_oemol() for posed_ligand in posed_ligands]
    filtered_results_idx = []
    for i, oemol1 in enumerate(all_oemols):

        # Check if this oemol is similar to any already selected
        for idx in filtered_results_idx:
            oemol2 = all_oemols[idx]
            if calculate_ligand_rmsd_oemol(oemol1, oemol2) <= cutoff:
                # Similar to an already selected mol so don't need this one
                break
        else:
            # if not similar to any in filtered_results_idx, add index to list
            filtered_results_idx.append(i)

    # pull Ligand objects from indices
    filtered_results = [posed_ligands[i] for i in filtered_results_idx]
    return filtered_results

In [ ]:
def make_df_from_docking_results(results=list[POSITDockingResults]):
    dfs = []
    for result in results:
        docking_dict = {}
        docking_dict["Query_Ligand"] = result.input_pair.ligand.compound_name
        docking_dict["Reference_Structure"] = (
            result.input_pair.complex.target.target_name
        )
        docking_dict["Reference_Ligand_SMILES"] = (
            result.input_pair.complex.ligand.smiles
        )
        docking_dict[DockingResultCols.SMILES.value] = result.input_pair.ligand.smiles
        docking_dict[DockingResultCols.DOCKING_CONFIDENCE_POSIT.value] = (
            result.posed_ligand.conf_tags["docking-confidence-POSIT"]
        )
        docking_dict["RMSD"] = result.posed_ligand.conf_tags["RMSD"]
        docking_dict["Pose_ID"] = result.posed_ligand.conf_tags["Pose_ID"]
        docking_dict["POSIT_Method"] = result.posed_ligand.conf_tags["_POSIT_method"]
        docking_dict["Reference_Ligand"] = (
            result.input_pair.complex.ligand.compound_name
        )

        dfs.append(pd.DataFrame(docking_dict))

    df = pd.concat(dfs)
    return df

In [ ]:
ligs = MolFileFactory(filename="/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ligand_files/combined_3d.sdf").load()

In [ ]:
lig_dict = {lig.compound_name: lig for lig in ligs}

In [ ]:
len(lig_dict.keys())

In [ ]:
docked_poses = MolFileFactory(filename="/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_multipose/399_docked/docking_results.sdf").load()

In [ ]:
docked_poses[-1]

In [ ]:
from collections import defaultdict

In [ ]:
pose_dict = defaultdict(list)

In [ ]:
_ = [pose_dict[lig.tags["ReferenceStructureName"]].append(lig) for lig in docked_poses]

In [ ]:
filtered_results = []
for target_name, posed_mols in pose_dict.items():
    filtered_results.extend(get_filtered_poses(posed_mols, cutoff=2)) 

In [ ]:
combined = Ligand.from_single_conformers(docked_poses)

In [ ]:
print(f"Calculating RMSD for {len(filtered_results)} poses")
records = []
for posed_lig in tqdm(filtered_results):
    ref = lig_dict[posed_lig.compound_name]
    calculate_ligand_rmsd(ref, posed_lig)
    records.append({"Query_Ligand": posed_lig.compound_name, 
                                    "Pose_ID": int(posed_lig.tags["Pose_ID"]), 
                                    "RMSD": posed_lig.tags["RMSD"],
                                    "Reference_Structure": posed_lig.tags["ReferenceStructureName"],
                                    "Reference_Ligand": posed_lig.tags["ReferenceLigandName"],
    "docking-confidence-POSIT": posed_lig.tags["docking-confidence-POSIT"],
    "POSIT_Method": posed_lig.tags["_POSIT_method"]
    })

In [ ]:
outdf = pd.DataFrame.from_records(records)

In [ ]:
og_df = pd.read_csv(Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_multipose/399_docked/docking_scores_raw.csv"))

In [ ]:
outdf = pd.DataFrame.from_records([{"Query_Ligand": lig.compound_name, 
                                    "Pose_ID": int(lig.tags["Pose_ID"]), 
                                    "RMSD": lig.tags["RMSD"],
                                    "Reference_Structure": lig.tags["ReferenceStructureName"],
                                    "Reference_Ligand": lig.tags["ReferenceLigandName"]} for lig in filtered_results])

In [ ]:
from harbor.analysis.cross_docking import DockingDataModel
import harbor.analysis.cross_docking as cd
from importlib import reload
reload(cd)

In [ ]:
ddm = cd.DockingDataModel.deserialize("test.parquet")

In [ ]:
ddm.dataframe

In [ ]:
kc = ddm.get_key_columns()

In [ ]:
kc = {item for item in kc for kc in ddm.key_columns_dict.values()}
pc = {item for pc in ddm.param_columns_dict.values() for item in pc}

In [ ]:
cc = kc.union(pc) - {"Pose_ID"}

In [ ]:
ddm.dataframe.groupby(list(cc)).head(2)

In [ ]:
cc = ddm.get_groupby_columns(except_cols=["Pose_ID"])

In [ ]:
ddm.dataframe.groupby(ddm.get_groupby_columns(["Pose_ID"])).head(2)